# Provision search: compression, quality and auto-merge benchmark (Google Colab)

This notebook benchmarks **our own provision embeddings** from Qdrant. It answers three questions:

1. **Bottleneck.** How is the live collection configured, and how fast and how accurate is the live search compared with an exact search?
2. **Quality.** Do the embeddings themselves look healthy (normalization, duplicates, boilerplate "hubs", how separated neighbours are from random pairs)? With an optional labelled test set, how good are the results?
3. **Compression and auto-merge.** What would scalar, binary or TurboQuant quantization do to recall, memory and speed on our data, and how many auto-merge decisions would flip at our threshold?

It is the companion of the doc *Provision search: bottleneck and quality checks*. It is separate from the TurboQuant workshop demos.

**Safety and data.**
* Everything against the production Qdrant is **read-only**: `get_collection`, `scroll` and `query_points`. Nothing is created, changed or deleted there.
* Section 7 creates collections only in a **temporary Qdrant server started inside this Colab runtime**, which disappears with the runtime.
* The sample of provision vectors (and, if `TEXT_FIELD` is set, some provision text) is downloaded into this Colab runtime. **Check this is allowed by our data policy before running.** Do not share an executed copy outside the team if it shows provision text.

**Credentials:** add `QDRANT_URL` and `QDRANT_API_KEY` in Colab › Secrets (key icon on the left). They are never printed.

**Runtime:** CPU is enough for up to about 100k vectors. A GPU runtime makes sections 3 to 5 faster.

In [ ]:
import subprocess, sys, importlib
for mod, spec in [("qdrant_client", "qdrant-client"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("requests", "requests")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", spec], check=True)
try:  # optional: TurboQuant with SIMD kernels
    import turbovec  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "turbovec"], check=False)

## Helper modules
The next two cells write `turboquant_core.py` (the TurboQuant reference implementation) and `provision_bench.py` (search, metrics and auto-merge helpers used below).

In [ ]:
%%writefile turboquant_core.py
"""
turboquant_core.py - a compact, pure-PyTorch reference implementation of TurboQuant.

Paper: Zandieh, Daliri, Hadian, Mirrokni. "TurboQuant: Online Vector Quantization with
Near-optimal Distortion Rate" (arXiv:2504.19874). Builds on QJL (arXiv:2406.03482) and
PolarQuant (arXiv:2502.02617).

What is implemented
  * Lloyd-Max codebooks for the exact coordinate distribution of a random unit vector
    (a scaled Beta distribution, Lemma 1 of the paper).
  * TurboQuant_mse  (Algorithm 1): random rotation -> per-coordinate optimal scalar quantizer.
  * TurboQuant_prod (Algorithm 2): TurboQuant_mse with b-1 bits + 1-bit QJL on the residual,
    giving an unbiased inner-product estimator.
  * Outlier-channel split ("2.5-bit" / "3.5-bit" configurations from Section 4.3).
  * Real bit-packing so memory numbers are measured, not estimated.
  * A uniform INT-b baseline with per-group fp16 scale + zero point (KIVI / HF QuantizedCache style).
  * A Hugging Face `transformers` (v5) KV-cache (`TurboQuantCache`) that stores keys/values
    compressed and dequantizes on read.

This is a readable reference, not a fused kernel: dequantization happens in PyTorch, so it
saves memory but does not speed attention up. Production speedups come from fused kernels
(e.g. vLLM `--kv-cache-dtype turboquant_*`).
"""
from __future__ import annotations

import math
from dataclasses import dataclass
from functools import lru_cache

import numpy as np
import torch

# ----------------------------------------------------------------------------------------
# 1. Codebooks: Lloyd-Max for the coordinate distribution of a uniformly random unit vector
# ----------------------------------------------------------------------------------------


@lru_cache(maxsize=None)
def lloyd_max_codebook(d: int, bits: int, iters: int = 400, grid: int = 200_001) -> np.ndarray:
    """Optimal scalar quantizer (2**bits centroids) for f(x) ~ (1 - x^2)^((d-3)/2) on [-1, 1].

    This solves the continuous 1-D k-means problem of Eq. (4) numerically. Results are cached.
    """
    k = 2 ** bits
    x = np.linspace(-1.0, 1.0, grid)[1:-1]
    logf = 0.5 * (d - 3) * np.log1p(-x * x) if d > 3 else np.zeros_like(x)
    f = np.exp(logf - logf.max())
    f /= f.sum()
    # initialise at quantiles
    cdf = np.cumsum(f)
    c = np.interp((np.arange(k) + 0.5) / k, cdf, x)
    for _ in range(iters):
        b = 0.5 * (c[1:] + c[:-1])
        cell = np.searchsorted(b, x)
        w = np.bincount(cell, weights=f, minlength=k)
        m = np.bincount(cell, weights=f * x, minlength=k)
        c_new = np.where(w > 0, m / np.maximum(w, 1e-300), c)
        if np.max(np.abs(c_new - c)) < 1e-12:
            c = c_new
            break
        c = c_new
    return c.astype(np.float64)


def codebook_mse_cost(d: int, bits: int) -> float:
    """d * C(f_X, b): expected ||x - Q^-1(Q(x))||^2 for a unit vector (Theorem 1)."""
    c = lloyd_max_codebook(d, bits)
    x = np.linspace(-1.0, 1.0, 200_001)[1:-1]
    logf = 0.5 * (d - 3) * np.log1p(-x * x)
    f = np.exp(logf - logf.max()); f /= f.sum()
    b = 0.5 * (c[1:] + c[:-1])
    err = (x - c[np.searchsorted(b, x)]) ** 2
    return float(d * (err * f).sum())


# ----------------------------------------------------------------------------------------
# 2. Bit packing (any bit-width; exactly bits*d/8 bytes per vector)
# ----------------------------------------------------------------------------------------


def pack_bits(idx: torch.Tensor, bits: int) -> torch.Tensor:
    """Pack integer codes in [0, 2**bits) of shape [..., d] into uint8 of shape [..., bits*d/8]."""
    d = idx.shape[-1]
    assert d % 8 == 0, "last dim must be a multiple of 8"
    idx = idx.to(torch.uint8)
    shifts = torch.arange(bits, device=idx.device, dtype=torch.uint8)
    planes = (idx.unsqueeze(-2) >> shifts.view(-1, 1)) & 1            # [..., bits, d]
    planes = planes.reshape(*idx.shape[:-1], bits, d // 8, 8)
    weights = (1 << torch.arange(8, device=idx.device, dtype=torch.uint8))
    packed = (planes * weights).sum(-1, dtype=torch.uint8)             # [..., bits, d/8]
    return packed.reshape(*idx.shape[:-1], bits * d // 8)


_LUT = {}


def _bit_lut(device):
    if device not in _LUT:
        _LUT[device] = ((torch.arange(256, device=device).unsqueeze(1) >> torch.arange(8, device=device)) & 1).to(torch.uint8)
    return _LUT[device]


def unpack_bits(packed: torch.Tensor, bits: int, d: int) -> torch.Tensor:
    """Inverse of pack_bits -> int64 codes of shape [..., d] (one table lookup per byte)."""
    planes = _bit_lut(packed.device)[packed.long()]                      # [..., bits*d/8, 8]
    planes = planes.reshape(*packed.shape[:-1], bits, d)
    codes = planes[..., 0, :].clone()
    for j in range(1, bits):
        codes |= planes[..., j, :] << j
    return codes.long()


# ----------------------------------------------------------------------------------------
# 3. TurboQuant quantizers
# ----------------------------------------------------------------------------------------


def random_rotation(d: int, seed: int, device=None) -> torch.Tensor:
    """Haar-random orthogonal matrix via QR of a Gaussian matrix (Section 3.1)."""
    g = torch.Generator().manual_seed(seed)
    a = torch.randn(d, d, generator=g, dtype=torch.float64)
    q, r = torch.linalg.qr(a)
    q = q * torch.sign(torch.diagonal(r)).unsqueeze(0)
    return q.to(device=device, dtype=torch.float32)


@dataclass
class Compressed:
    """A batch of compressed vectors. Every field is a tensor with the same leading shape."""
    idx: torch.Tensor                    # packed MSE codes (uint8)
    norm: torch.Tensor                   # ||x|| in fp16
    qjl: torch.Tensor | None = None      # packed QJL sign bits (uint8), prod mode only
    gamma: torch.Tensor | None = None    # ||residual|| in fp16, prod mode only

    def nbytes(self) -> int:
        return sum(t.numel() * t.element_size() for t in (self.idx, self.norm, self.qjl, self.gamma) if t is not None)

    def cat(self, other: "Compressed", dim: int) -> "Compressed":
        def c(a, b):
            return None if a is None else torch.cat([a, b], dim=dim)
        return Compressed(c(self.idx, other.idx), c(self.norm, other.norm), c(self.qjl, other.qjl), c(self.gamma, other.gamma))


class TurboQuant:
    """TurboQuant for d-dimensional vectors (arbitrary norm; the norm is stored in fp16).

    mode="mse"  -> Algorithm 1, `bits` bits per coordinate.
    mode="prod" -> Algorithm 2, (bits-1)-bit MSE stage + 1-bit QJL on the residual (unbiased <y, x>).
    renorm=True -> rescale the reconstruction so its norm equals the stored norm (cheap "norm
                   correction", similar in spirit to vLLM's *_nc variants).
    """

    def __init__(self, d: int, bits: int, mode: str = "mse", seed: int = 0, device=None, renorm: bool = False):
        assert mode in ("mse", "prod")
        assert d % 8 == 0
        self.d, self.bits, self.mode, self.renorm = d, bits, mode, renorm
        self.mse_bits = bits if mode == "mse" else bits - 1
        assert self.mse_bits >= 0
        self.device = device
        self.Pi = random_rotation(d, seed, device)                      # rotation (d x d)
        if self.mse_bits > 0:
            c = torch.tensor(lloyd_max_codebook(d, self.mse_bits), dtype=torch.float32, device=device)
            self.centroids = c
            self.boundaries = 0.5 * (c[1:] + c[:-1])
        if mode == "prod":
            g = torch.Generator().manual_seed(seed + 7919)
            self.S = torch.randn(d, d, generator=g).to(device)          # QJL projection
        self.bits_per_vector = bits * d + 16 + (16 if mode == "prod" else 0)

    def to(self, device):
        self.device = device
        for name in ("Pi", "centroids", "boundaries", "S"):
            if hasattr(self, name):
                setattr(self, name, getattr(self, name).to(device))
        return self

    # --- Algorithm 1 -------------------------------------------------------------------
    def _mse_codes(self, u: torch.Tensor) -> torch.Tensor:
        y = u @ self.Pi.T
        return torch.bucketize(y, self.boundaries)

    def _mse_decode(self, codes: torch.Tensor) -> torch.Tensor:
        return self.centroids[codes] @ self.Pi

    # --- public API -------------------------------------------------------------------
    @torch.no_grad()
    def quantize(self, x: torch.Tensor) -> Compressed:
        x = x.float()
        norm = x.norm(dim=-1, keepdim=True)
        u = x / norm.clamp_min(1e-12)
        if self.mse_bits > 0:
            codes = self._mse_codes(u)
            idx = pack_bits(codes, self.mse_bits)
            u_mse = self._mse_decode(codes)
        else:
            idx = torch.zeros(*x.shape[:-1], 0, dtype=torch.uint8, device=x.device)
            u_mse = torch.zeros_like(u)
        if self.mode == "mse":
            return Compressed(idx, norm.squeeze(-1).half())
        r = u - u_mse                                                    # residual
        signs = (r @ self.S.T) >= 0
        return Compressed(idx, norm.squeeze(-1).half(), pack_bits(signs, 1), r.norm(dim=-1).half())

    @torch.no_grad()
    def dequantize(self, c: Compressed, dtype=torch.float32) -> torch.Tensor:
        if self.mse_bits > 0:
            u = self._mse_decode(unpack_bits(c.idx, self.mse_bits, self.d))
        else:
            u = torch.zeros(*c.norm.shape, self.d, device=c.norm.device)
        if self.mode == "prod":
            z = unpack_bits(c.qjl, 1, self.d).float() * 2 - 1              # {-1,+1}
            u = u + (math.sqrt(math.pi / 2) / self.d) * c.gamma.float().unsqueeze(-1) * (z @ self.S)
        if self.renorm:
            u = u / u.norm(dim=-1, keepdim=True).clamp_min(1e-12)
        return (u * c.norm.float().unsqueeze(-1)).to(dtype)

    def roundtrip(self, x: torch.Tensor) -> torch.Tensor:
        return self.dequantize(self.quantize(x), x.dtype)


class MixedTurboQuant:
    """Outlier split from Section 4.3: `n_out` outlier channels get `bits_hi`, the rest `bits_lo`.

    Example (paper): head_dim 128, 32 outliers @3 bits + 96 @2 bits = 2.5 bits/channel.
    Outlier channels are picked by `calibrate` (largest mean |x| on a sample, e.g. the prefill).
    """

    def __init__(self, d: int, n_out: int, bits_hi: int, bits_lo: int, mode="mse", seed=0, device=None, renorm=False):
        self.d, self.n_out = d, n_out
        self.hi = TurboQuant(n_out, bits_hi, mode, seed, device, renorm)
        self.lo = TurboQuant(d - n_out, bits_lo, mode, seed + 1, device, renorm)
        self.out_idx = torch.arange(n_out, device=device)
        self.in_idx = torch.arange(n_out, d, device=device)
        self.bits = (n_out * bits_hi + (d - n_out) * bits_lo) / d
        self.calibrated = False

    def calibrate(self, sample: torch.Tensor):
        score = sample.float().abs().reshape(-1, self.d).mean(0)
        order = torch.argsort(score, descending=True)
        self.out_idx, self.in_idx = order[: self.n_out].sort().values, order[self.n_out:].sort().values
        self.calibrated = True

    def to(self, device):
        self.hi.to(device); self.lo.to(device)
        self.out_idx, self.in_idx = self.out_idx.to(device), self.in_idx.to(device)
        return self

    def quantize(self, x):
        if not self.calibrated:
            self.calibrate(x)
        return (self.hi.quantize(x[..., self.out_idx]), self.lo.quantize(x[..., self.in_idx]))

    def dequantize(self, c, dtype=torch.float32):
        a, b = c
        lead = a.norm.shape
        out = torch.empty(*lead, self.d, device=a.norm.device, dtype=dtype)
        out[..., self.out_idx] = self.hi.dequantize(a, dtype)
        out[..., self.in_idx] = self.lo.dequantize(b, dtype)
        return out

    def roundtrip(self, x):
        return self.dequantize(self.quantize(x), x.dtype)


class UniformQuant:
    """Baseline: asymmetric min-max INT-b per group of `group` channels, fp16 scale + zero per group.

    Effective bits/channel = bits + 32/group (the "memory overhead" TurboQuant removes).
    """

    def __init__(self, d: int, bits: int, group: int = 32):
        assert d % group == 0
        self.d, self.bits, self.group = d, bits, group
        self.bits_per_vector = bits * d + 32 * (d // group)

    def to(self, device):
        return self

    @torch.no_grad()
    def quantize(self, x):
        g = x.float().reshape(*x.shape[:-1], self.d // self.group, self.group)
        lo, hi = g.amin(-1, keepdim=True), g.amax(-1, keepdim=True)
        scale = ((hi - lo) / (2 ** self.bits - 1)).clamp_min(1e-8)
        q = torch.round((g - lo) / scale).clamp(0, 2 ** self.bits - 1)
        packed = pack_bits(q.reshape(*x.shape[:-1], self.d).long(), self.bits)
        return (packed, scale.half(), lo.half())

    @torch.no_grad()
    def dequantize(self, c, dtype=torch.float32):
        packed, scale, lo = c
        q = unpack_bits(packed, self.bits, self.d).float()
        q = q.reshape(*q.shape[:-1], self.d // self.group, self.group)
        return (q * scale.float() + lo.float()).reshape(*q.shape[:-2], self.d).to(dtype)

    def roundtrip(self, x):
        return self.dequantize(self.quantize(x), x.dtype)


def nbytes(c) -> int:
    """Bytes used by any compressed object returned by the quantizers above."""
    if isinstance(c, Compressed):
        return c.nbytes()
    if isinstance(c, (tuple, list)):
        return sum(nbytes(t) for t in c)
    if isinstance(c, torch.Tensor):
        return c.numel() * c.element_size()
    return 0


def cat_compressed(a, b, dim):
    if isinstance(a, Compressed):
        return a.cat(b, dim)
    if isinstance(a, (tuple, list)):
        return type(a)(cat_compressed(x, y, dim) for x, y in zip(a, b))
    return torch.cat([a, b], dim=dim)


# ----------------------------------------------------------------------------------------
# 4. Hugging Face transformers (v5) KV cache
# ----------------------------------------------------------------------------------------
try:
    from transformers.cache_utils import Cache, DynamicLayer

    class TurboQuantLayer(DynamicLayer):
        """One attention layer's cache: keys/values stored compressed, the last
        `residual_length` tokens kept in full precision (0 = quantize everything, as in the paper)."""

        def __init__(self, k_quant, v_quant, residual_length: int = 0):
            super().__init__()
            self.kq, self.vq = k_quant, v_quant
            self.residual_length = residual_length
            self.ck = self.cv = None          # compressed storage, token axis = -2 of the original
            self.cumulative_length = 0

        def lazy_initialization(self, key_states, value_states):
            super().lazy_initialization(key_states, value_states)
            for q in (self.kq, self.vq):
                q.to(key_states.device)
                if isinstance(q, MixedTurboQuant) and not q.calibrated:
                    q.calibrate(key_states if q is self.kq else value_states)

        def _append(self, k, v):
            ck, cv = self.kq.quantize(k), self.vq.quantize(v)
            if self.ck is None:
                self.ck, self.cv = ck, cv
            else:  # leading dims [B, H, T] -> token axis is 2 for every stored field
                self.ck, self.cv = cat_compressed(self.ck, ck, 2), cat_compressed(self.cv, cv, 2)

        def update(self, key_states, value_states, *args, **kwargs):
            if not self.is_initialized:
                self.lazy_initialization(key_states, value_states)
            self.cumulative_length += key_states.shape[-2]
            dt = key_states.dtype
            parts_k, parts_v = [], []
            if self.ck is not None:
                parts_k.append(self.kq.dequantize(self.ck, dt))
                parts_v.append(self.vq.dequantize(self.cv, dt))
            self.keys = torch.cat([self.keys, key_states], dim=-2)
            self.values = torch.cat([self.values, value_states], dim=-2)
            k_out = torch.cat(parts_k + [self.keys], dim=-2)
            v_out = torch.cat(parts_v + [self.values], dim=-2)
            overflow = self.keys.shape[-2] - self.residual_length
            if overflow > 0:  # move the oldest tokens from the fp16 buffer into compressed storage
                self._append(self.keys[..., :overflow, :], self.values[..., :overflow, :])
                self.keys = self.keys[..., overflow:, :].contiguous()
                self.values = self.values[..., overflow:, :].contiguous()
            return k_out, v_out

        def get_seq_length(self) -> int:
            return self.cumulative_length

        def reset(self):
            self.ck = self.cv = None
            self.cumulative_length = 0
            super().reset()

        def nbytes(self) -> int:
            fp = 0 if self.keys is None else self.keys.numel() * self.keys.element_size() * 2
            return fp + nbytes(self.ck) + nbytes(self.cv)

    class TurboQuantCache(Cache):
        """Drop-in KV cache for HF `generate`/forward.

        make_k / make_v: callables (layer_idx, head_dim) -> quantizer (TurboQuant, MixedTurboQuant,
        UniformQuant). Each layer gets its own random rotation via the seed.
        """

        def __init__(self, config, make_k, make_v, residual_length: int = 0):
            cfg = config.get_text_config(decoder=True) if hasattr(config, "get_text_config") else config
            hd = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
            layers = [TurboQuantLayer(make_k(i, hd), make_v(i, hd), residual_length) for i in range(cfg.num_hidden_layers)]
            super().__init__(layers=layers)

        def nbytes(self) -> int:
            return sum(l.nbytes() for l in self.layers)

    def fp16_cache_nbytes(cache) -> int:
        """Bytes of a regular DynamicCache."""
        return sum(l.keys.numel() * l.keys.element_size() + l.values.numel() * l.values.element_size()
                   for l in cache.layers if l.keys is not None)

except ImportError:  # transformers not installed: quantizers still work
    pass


# ----------------------------------------------------------------------------------------
# 5. Convenience factories
# ----------------------------------------------------------------------------------------

def tq_factory(bits, mode="mse", renorm=False, seed=0):
    def make(layer_idx, d):
        return TurboQuant(d, bits, mode=mode, seed=seed + 1000 * layer_idx, renorm=renorm)
    return make


def mixed_factory(n_out_frac, bits_hi, bits_lo, mode="mse", renorm=False, seed=0):
    def make(layer_idx, d):
        n_out = int(round(d * n_out_frac / 8)) * 8
        return MixedTurboQuant(d, n_out, bits_hi, bits_lo, mode=mode, seed=seed + 1000 * layer_idx, renorm=renorm)
    return make


def uniform_factory(bits, group=32):
    def make(layer_idx, d):
        return UniformQuant(d, bits, group)
    return make


In [ ]:
%%writefile provision_bench.py
"""
provision_bench.py - helpers for benchmarking vector compression on our own embeddings.

Used by provision_search_benchmark.ipynb (which embeds this file with %%writefile).
Pure numpy + torch; turbovec is optional. Nothing here talks to Qdrant.

Conventions
  * X: float32 [n, d] database vectors, already normalized when the metric is cosine.
  * Queries are database points ("find provisions similar to this provision"), given as
    row indices `qidx`; each query's own row is excluded from its results.
  * Scores are inner products (cosine for normalized vectors). Higher = more similar.
"""
from __future__ import annotations

import math
import time

import numpy as np
import torch

from turboquant_core import TurboQuant, unpack_bits

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ----------------------------------------------------------------------------------------
# exact search and metrics
# ----------------------------------------------------------------------------------------


def normalize(X: np.ndarray) -> np.ndarray:
    return (X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)).astype(np.float32)


def _topk_excluding_self(score: torch.Tensor, qidx_chunk: torch.Tensor, offset: int, k: int):
    """Mask each query's own row (if it falls inside this database chunk) and take top-k."""
    rows = torch.arange(score.shape[0], device=score.device)
    local = qidx_chunk - offset
    inside = (local >= 0) & (local < score.shape[1])
    score[rows[inside], local[inside]] = -float("inf")
    kk = min(k, score.shape[1])
    s, i = score.topk(kk, dim=1)
    return s, i + offset


def merge_topk(best_s, best_i, s, i, k):
    cs, ci = torch.cat([best_s, s], 1), torch.cat([best_i, i], 1)
    s2, top = cs.topk(k, dim=1)
    return s2, ci.gather(1, top)


@torch.no_grad()
def exact_topk(X: np.ndarray, qidx: np.ndarray, k: int, chunk: int = 50_000, qbatch: int = 1024):
    """Exact inner-product top-k for the query rows, self excluded. Returns (scores, indices) numpy."""
    Xt = torch.from_numpy(X).to(DEVICE)
    out_s, out_i = [], []
    for a in range(0, len(qidx), qbatch):
        qi = torch.from_numpy(qidx[a:a + qbatch]).to(DEVICE)
        q = Xt[qi]
        best_s = torch.full((len(qi), k), -float("inf"), device=DEVICE)
        best_i = torch.full((len(qi), k), -1, dtype=torch.long, device=DEVICE)
        for s0 in range(0, Xt.shape[0], chunk):
            sc = q @ Xt[s0:s0 + chunk].T
            s, i = _topk_excluding_self(sc, qi, s0, k)
            best_s, best_i = merge_topk(best_s, best_i, s, i, k)
        out_s.append(best_s.cpu()); out_i.append(best_i.cpu())
    return torch.cat(out_s).numpy(), torch.cat(out_i).numpy()


def recall_at_k(I_approx: np.ndarray, I_true: np.ndarray, k: int) -> float:
    """Average overlap between the true top-k and the returned top-k (k@k)."""
    return float(np.mean([len(set(a[:k]) & set(t[:k])) / k for a, t in zip(I_approx, I_true)]))


def r1_at_k(I_approx: np.ndarray, I_true: np.ndarray, k: int) -> float:
    """How often the true nearest neighbour is inside the returned top-k (the paper's Recall@1@k)."""
    return float(np.mean([t[0] in set(a[:k]) for a, t in zip(I_approx, I_true)]))


# ----------------------------------------------------------------------------------------
# compression methods (each mimics a Qdrant quantization option, scored asymmetrically:
# the query stays in float32, only stored vectors are compressed, as Qdrant does)
# ----------------------------------------------------------------------------------------


class Method:
    name = "base"
    bits = 32.0
    reconstructs = False

    def build(self, X: np.ndarray):
        raise NotImplementedError

    def scores(self, q: torch.Tensor, s0: int, s1: int) -> torch.Tensor:
        """Approximate scores of float queries q [m, d] against stored rows s0:s1."""
        raise NotImplementedError

    def reconstruct(self, idx: np.ndarray) -> np.ndarray:
        raise NotImplementedError

    def nbytes(self) -> int:
        raise NotImplementedError

    @torch.no_grad()
    def search(self, X: np.ndarray, qidx: np.ndarray, k: int, chunk: int = 50_000, qbatch: int = 1024):
        Xt = torch.from_numpy(X).to(DEVICE)
        out_s, out_i = [], []
        for a in range(0, len(qidx), qbatch):
            qi = torch.from_numpy(qidx[a:a + qbatch]).to(DEVICE)
            q = Xt[qi]
            best_s = torch.full((len(qi), k), -float("inf"), device=DEVICE)
            best_i = torch.full((len(qi), k), -1, dtype=torch.long, device=DEVICE)
            for s0 in range(0, self.n, chunk):
                sc = self.scores(q, s0, min(s0 + chunk, self.n))
                s, i = _topk_excluding_self(sc, qi, s0, k)
                best_s, best_i = merge_topk(best_s, best_i, s, i, k)
            out_s.append(best_s.cpu()); out_i.append(best_i.cpu())
        return torch.cat(out_s).numpy(), torch.cat(out_i).numpy()


class Float32(Method):
    name, bits, reconstructs = "float32 (no quantization)", 32.0, True

    def build(self, X):
        self.X = torch.from_numpy(X).to(DEVICE); self.n = len(X); return self

    def scores(self, q, s0, s1):
        return q @ self.X[s0:s1].T

    def reconstruct(self, idx):
        return self.X[torch.from_numpy(idx).to(DEVICE)].cpu().numpy()

    def nbytes(self):
        return self.X.numel() * 4


class ScalarInt8(Method):
    """Like Qdrant scalar quantization (int8): one global range from the `quantile` of all values."""
    bits, reconstructs = 8.0, True

    def __init__(self, quantile: float = 0.99):
        self.quantile = quantile
        self.name = f"scalar int8 (quantile {quantile})"

    def build(self, X):
        flat = X.reshape(-1)
        sample = flat[np.random.default_rng(0).choice(flat.size, min(flat.size, 2_000_000), replace=False)]
        lo, hi = np.quantile(sample, [(1 - self.quantile) / 2, 1 - (1 - self.quantile) / 2])
        self.lo, self.step = float(lo), float(hi - lo) / 255
        codes = np.clip(np.round((X - self.lo) / self.step), 0, 255).astype(np.uint8)
        self.codes = torch.from_numpy(codes).to(DEVICE); self.n = len(X)
        return self

    def _deq(self, s0, s1):
        return self.codes[s0:s1].float() * self.step + self.lo

    def scores(self, q, s0, s1):
        return q @ self._deq(s0, s1).T

    def reconstruct(self, idx):
        return (self.codes[torch.from_numpy(idx).to(DEVICE)].float() * self.step + self.lo).cpu().numpy()

    def nbytes(self):
        return self.codes.numel()


class Binary1(Method):
    """Like Qdrant binary quantization: one sign bit per dimension, query kept in float."""
    name, bits, reconstructs = "binary 1-bit", 1.0, False

    def build(self, X):
        self.packed = torch.from_numpy(np.packbits(X > 0, axis=1)).to(DEVICE)
        self.d, self.n = X.shape[1], len(X)
        return self

    def scores(self, q, s0, s1):
        bits = torch.from_numpy(np.unpackbits(self.packed[s0:s1].cpu().numpy(), axis=1)[:, :self.d]).to(DEVICE)
        return q @ (bits.float() * 2 - 1).T

    def nbytes(self):
        return self.packed.numel()


class TurboQuantRef(Method):
    """TurboQuant_mse from turboquant_core (random rotation + Lloyd-Max codebook), optional norm correction.

    renorm=True rescales each reconstructed vector to its stored norm, similar in spirit to the length
    renormalization Qdrant adds in its own TurboQuant implementation.
    """
    reconstructs = True

    def __init__(self, bits: int, renorm: bool = True, seed: int = 0):
        self.bits, self.renorm, self.seed = float(bits), renorm, seed
        self.name = f"TurboQuant {bits}-bit" + (" + norm correction" if renorm else "")

    def build(self, X, batch: int = 50_000):
        d = X.shape[1]
        assert d % 8 == 0, "TurboQuant reference needs a dimension divisible by 8"
        self.q = TurboQuant(d, int(self.bits), "mse", seed=self.seed, device=DEVICE, renorm=self.renorm)
        parts = [self.q.quantize(torch.from_numpy(X[i:i + batch]).to(DEVICE)) for i in range(0, len(X), batch)]
        self.idx = torch.cat([p.idx for p in parts]); self.norm = torch.cat([p.norm for p in parts])
        self.d, self.n = d, len(X)
        # renorm factor = ||x|| / ||c[idx]|| (the rotation keeps lengths, so it is computed in the rotated space)
        if self.renorm:
            cn = []
            for s0 in range(0, self.n, batch):
                y = self.q.centroids[unpack_bits(self.idx[s0:s0 + batch], int(self.bits), d)]
                cn.append(y.norm(dim=1))
            self.scale = self.norm.float() / torch.cat(cn).clamp_min(1e-12)
        else:
            self.scale = self.norm.float()
        return self

    def scores(self, q, s0, s1):
        y = self.q.centroids[unpack_bits(self.idx[s0:s1], int(self.bits), self.d)]   # rotated reconstruction
        return ((q @ self.q.Pi.T) @ y.T) * self.scale[s0:s1]

    def reconstruct(self, idx):
        ii = torch.from_numpy(idx).to(DEVICE)
        y = self.q.centroids[unpack_bits(self.idx[ii], int(self.bits), self.d)]
        return ((y @ self.q.Pi) * self.scale[ii].unsqueeze(1)).cpu().numpy()

    def nbytes(self):
        return self.idx.numel() + self.norm.numel() * 2


class Turbovec(Method):
    """turbovec (Rust + SIMD TurboQuant). Optional: only if `pip install turbovec` worked."""
    reconstructs = False

    def __init__(self, bits: int):
        self.bits = float(bits); self.name = f"turbovec {bits}-bit (SIMD)"

    def build(self, X):
        import turbovec
        self.ix = turbovec.TurboQuantIndex(dim=X.shape[1], bit_width=int(self.bits))
        self.ix.add(X); self.ix.prepare(); self.n = len(X)
        return self

    def search(self, X, qidx, k, **_):
        s, i = self.ix.search(np.ascontiguousarray(X[qidx]), k + 1)
        s, i = np.asarray(s), np.asarray(i).astype(np.int64)
        return drop_self(s, i, qidx, k)

    def nbytes(self):
        return len(self.ix.to_bytes())


def drop_self(S: np.ndarray, I: np.ndarray, qidx: np.ndarray, k: int):
    """Remove each query's own index from a (k+1)-wide result and keep k columns."""
    keep = I != qidx[:, None]
    S2 = np.stack([s[m][:k] for s, m in zip(S, keep)]); I2 = np.stack([i[m][:k] for i, m in zip(I, keep)])
    return S2, I2


def rescore(X: np.ndarray, qidx: np.ndarray, I_cand: np.ndarray, k: int):
    """Re-rank candidate lists with exact float32 scores (what Qdrant's `rescore=True` does)."""
    q = X[qidx]
    S = np.einsum("qd,qcd->qc", q, X[I_cand])
    order = np.argsort(-S, axis=1)[:, :k]
    return np.take_along_axis(S, order, 1), np.take_along_axis(I_cand, order, 1)


# ----------------------------------------------------------------------------------------
# benchmark driver
# ----------------------------------------------------------------------------------------


def run_method(method: Method, X, qidx, I_true, k=10, oversampling=(1, 2, 4), raw_bytes=None):
    """Build once, then search without and with rescoring at each oversampling factor."""
    raw_bytes = raw_bytes or X.nbytes
    t = time.time(); method.build(X)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    build_s = time.time() - t
    rows = []
    for ov in oversampling:
        if isinstance(method, Float32) and ov > 1:
            continue  # exact scores already; rescoring changes nothing
        t = time.time(); S, I = method.search(X, qidx, k * ov)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        search_s = time.time() - t
        if ov == 1:
            variants = [("no rescore", I[:, :k], search_s)]
        else:
            t = time.time(); _, I2 = rescore(X, qidx, I, k)
            variants = [(f"rescore, oversampling {ov}x", I2, search_s + time.time() - t)]
        for label, Ik, sec in variants:
            rows.append(dict(method=method.name, search=label, bits_per_dim=method.bits,
                             MB=round(method.nbytes() / 2 ** 20, 1), compression=round(raw_bytes / method.nbytes(), 1),
                             build_s=round(build_s, 2), QPS=round(len(qidx) / max(sec, 1e-9)),
                             **{f"recall@{k}": round(recall_at_k(Ik, I_true, k), 4),
                                "top1_found_in_top1": round(r1_at_k(Ik, I_true, 1), 4),
                                f"top1_found_in_top{k}": round(r1_at_k(Ik, I_true, k), 4)}))
    return rows


# ----------------------------------------------------------------------------------------
# embedding health (quality diagnostics that need no labels)
# ----------------------------------------------------------------------------------------


def embedding_health(X_raw: np.ndarray, X: np.ndarray, S_true: np.ndarray, I_true: np.ndarray, n_pairs: int = 20_000):
    g = np.random.default_rng(0)
    norms = np.linalg.norm(X_raw, axis=1)
    a, b = g.integers(0, len(X), n_pairs), g.integers(0, len(X), n_pairs)
    rand_cos = np.einsum("nd,nd->n", X[a], X[b])
    mean_dir = float(np.linalg.norm(X.mean(0)))
    # effective dimension: components needed for 90% of the variance (on a sample)
    samp = X[g.choice(len(X), min(len(X), 20_000), replace=False)]
    ev = np.linalg.svd(samp - samp.mean(0), compute_uv=False) ** 2
    eff90 = int(np.searchsorted(np.cumsum(ev) / ev.sum(), 0.90) + 1)
    # duplicates and hubs
    rounded = np.round(X, 5)
    _, counts = np.unique(rounded, axis=0, return_counts=True)
    exact_dups = int((counts - 1).clip(min=0).sum())
    occ = np.bincount(I_true[:, :10].reshape(-1), minlength=len(X))
    hub_share = float(np.sort(occ)[::-1][: max(1, len(X) // 100)].sum() / max(occ.sum(), 1))
    return dict(
        vectors=len(X), dim=X.shape[1],
        norm_min=float(norms.min()), norm_median=float(np.median(norms)), norm_max=float(norms.max()),
        already_normalized=bool(np.allclose(norms, 1, atol=1e-3)),
        random_pair_cos_mean=float(rand_cos.mean()), random_pair_cos_p95=float(np.quantile(rand_cos, 0.95)),
        mean_direction_norm=mean_dir, effective_dim_90pct=eff90,
        exact_duplicate_vectors=exact_dups,
        nn1_score_median=float(np.median(S_true[:, 0])),
        near_duplicate_queries_pct=float((S_true[:, 0] > 0.99).mean() * 100),
        top1pct_hubs_share_of_top10_slots=hub_share,
    ), rand_cos, occ


# ----------------------------------------------------------------------------------------
# auto-merge analysis
# ----------------------------------------------------------------------------------------


def candidate_pairs(qidx: np.ndarray, S_true: np.ndarray, I_true: np.ndarray, min_score: float):
    """Pairs (a, b, exact score) among each query's exact top-k with score >= min_score, deduplicated."""
    seen, rows = set(), []
    for q, srow, irow in zip(qidx, S_true, I_true):
        for s, j in zip(srow, irow):
            if s < min_score:
                break
            key = (min(q, j), max(q, j))
            if key not in seen:
                seen.add(key); rows.append((key[0], key[1], float(s)))
    if not rows:
        return np.zeros(0, np.int64), np.zeros(0, np.int64), np.zeros(0, np.float32)
    a, b, s = map(np.array, zip(*rows))
    return a.astype(np.int64), b.astype(np.int64), s.astype(np.float32)


def approx_pair_scores(method: Method, X: np.ndarray, a: np.ndarray, b: np.ndarray, batch: int = 50_000):
    """Score of each pair as a quantized search would see it: float query a against stored (compressed) b."""
    out = []
    for s0 in range(0, len(a), batch):
        rb = method.reconstruct(b[s0:s0 + batch])
        out.append(np.einsum("nd,nd->n", X[a[s0:s0 + batch]], rb))
    return np.concatenate(out) if out else np.zeros(0, np.float32)


def merge_flips(exact: np.ndarray, approx: np.ndarray, threshold: float) -> dict:
    e, p = exact >= threshold, approx >= threshold
    return dict(pairs_checked=int(len(exact)), merges_exact=int(e.sum()),
                false_merges=int((p & ~e).sum()), missed_merges=int((e & ~p).sum()),
                score_error_mean=float(np.mean(approx - exact)) if len(exact) else 0.0,
                score_error_p95_abs=float(np.quantile(np.abs(approx - exact), 0.95)) if len(exact) else 0.0)


# ----------------------------------------------------------------------------------------
# labelled evaluation (optional CSVs from domain experts)
# ----------------------------------------------------------------------------------------


def recall_from_labels(I: np.ndarray, qidx: np.ndarray, relevant: dict, k: int = 10) -> float:
    """relevant: {query row index -> set of relevant row indices}. Average share of them found in the top-k."""
    vals = []
    for q, row in zip(qidx, I):
        rel = relevant.get(int(q))
        if rel:
            vals.append(len(set(row[:k]) & rel) / len(rel))
    return float(np.mean(vals)) if vals else float("nan")


def merge_pr_curve(scores: np.ndarray, same: np.ndarray, thresholds: np.ndarray):
    """Precision and recall of 'merge if score >= t' against labelled pairs (same = 1 for true duplicates)."""
    rows = []
    for t in thresholds:
        pred = scores >= t
        tp = int((pred & (same == 1)).sum()); fp = int((pred & (same == 0)).sum()); fn = int((~pred & (same == 1)).sum())
        rows.append(dict(threshold=float(t), precision=tp / max(tp + fp, 1), recall=tp / max(tp + fn, 1),
                         merges=int(pred.sum()), wrong_merges=fp))
    return rows


## Settings
Edit this cell, then Runtime › Run all.

In [ ]:
import os
SOURCE = "qdrant"            # "qdrant" = read a sample from the live collection; "npy" = load NPY_VECTORS / NPY_IDS instead
COLLECTION = "provisions"     # name of the provisions collection (check it in the Qdrant dashboard)
VECTOR_NAME = None            # set to the vector name if the collection uses named vectors, e.g. "dense"
TEXT_FIELD = None             # payload field with the provision text, e.g. "text"; None = no text is downloaded or shown
N_SAMPLE = 100_000            # vectors to pull for the offline benchmark (more = slower, closer to production)
N_QUERIES = 1_000             # provisions used as queries ("find provisions similar to this one")
K = 10                        # results shown by "similar provisions"
MERGE_THRESHOLD = 0.90        # the similarity threshold the auto-merge job uses (put the real value here)
N_LIVE_QUERIES = 100          # queries sent to the live collection in section 6 (read-only)
RUN_LOCAL_QDRANT = True       # section 7: start a temporary Qdrant 1.18 inside Colab and test its quantization options
QDRANT_VERSION = "v1.18.0"    # release used in section 7 (needs 1.18 or later for TurboQuant)
NPY_VECTORS, NPY_IDS = "provision_vectors.npy", "provision_ids.npy"   # only for SOURCE = "npy"
SIMILAR_LABELS_CSV = "similar_labels.csv"   # optional, section 8: columns query_id, relevant_id
MERGE_LABELS_CSV = "merge_labels.csv"       # optional, section 8: columns id_a, id_b, same (1 = same provision)


def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


QDRANT_URL, QDRANT_API_KEY = secret("QDRANT_URL"), secret("QDRANT_API_KEY")
print("Qdrant URL set:", bool(QDRANT_URL), "| API key set:", bool(QDRANT_API_KEY))

In [ ]:
import math, time, json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
import turboquant_core, provision_bench as pb
importlib.reload(turboquant_core); importlib.reload(pb)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200)
print("device:", pb.DEVICE)
RESULTS = {}   # tables saved at the end

## 1. The live collection (read-only)
Configuration that matters for speed and quality, with automatic flags for the checks in the doc's "Bottleneck checks" section.

In [ ]:
client = None
DIST = "Cosine"
if SOURCE == "qdrant":
    from qdrant_client import QdrantClient, models
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)
    info = client.get_collection(COLLECTION)
    cfg = info.config
    vparams = cfg.params.vectors
    if isinstance(vparams, dict):  # named vectors
        assert VECTOR_NAME in vparams, f"named vectors {list(vparams)}: set VECTOR_NAME to one of them"
        vparams = vparams[VECTOR_NAME]
    DIST = str(vparams.distance.value if hasattr(vparams.distance, "value") else vparams.distance)
    hnsw = vparams.hnsw_config or cfg.hnsw_config
    quant = vparams.quantization_config or cfg.quantization_config
    live = dict(status=str(info.status), points=info.points_count, indexed_vectors=info.indexed_vectors_count,
                segments=info.segments_count, dim=vparams.size, distance=DIST,
                vectors_on_disk=getattr(vparams, "on_disk", None), hnsw_m=hnsw.m, hnsw_ef_construct=hnsw.ef_construct,
                hnsw_on_disk=hnsw.on_disk, quantization=json.dumps(quant.model_dump(exclude_none=True)) if quant else None,
                payload_indexes=", ".join(f"{k} ({v.data_type})" for k, v in (info.payload_schema or {}).items()) or None)
    display(pd.DataFrame([live]).T.rename(columns={0: "value"}))
    flags = []
    if "green" not in live["status"].lower():
        flags.append("Status is not green: the optimizer is still working, searches may be slower.")
    if live["points"] and live["indexed_vectors"] is not None and live["indexed_vectors"] < 0.9 * live["points"]:
        flags.append(f"Only {live['indexed_vectors']:,} of {live['points']:,} vectors are in the HNSW index: part of the data is searched without it.")
    if live["vectors_on_disk"] and not quant:
        flags.append("Vectors are on disk with no quantized copy in RAM: every search reads from disk.")
    if live["segments"] and live["points"] and live["segments"] > 32 and live["points"] / live["segments"] < 20_000:
        flags.append(f"{live['segments']} segments for {live['points']:,} points: many small segments slow every search.")
    if not live["payload_indexes"]:
        flags.append("No payload indexes: any filtered search scans payloads.")
    if quant:
        flags.append("Quantization is on: make sure the code searches with rescore enabled (section 6 measures the effect).")
    print("\n".join("FLAG: " + f for f in flags) or "No configuration flags.")
    RESULTS["live_collection"] = pd.DataFrame([live])
assert DIST.lower().startswith(("cos", "dot")), "This notebook handles Cosine and Dot collections; for Euclid, ask before using it."

## 2. Load a sample of provision vectors
Pulled with `scroll` (read-only) in id order and cached in this runtime. Note that the first `N_SAMPLE` points by id may not be a random sample of the collection.

In [ ]:
TEXTS = None
if SOURCE == "qdrant":
    cache = f"/content/{COLLECTION}_{N_SAMPLE}.npz" if os.path.isdir("/content") else f"{COLLECTION}_{N_SAMPLE}.npz"
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True); X_raw, IDS = z["X"], z["ids"]; TEXTS = z["texts"] if "texts" in z else None
    else:
        vecs, ids, texts, offset, t0 = [], [], [], None, time.time()
        while len(ids) < N_SAMPLE:
            pts, offset = client.scroll(COLLECTION, limit=min(1000, N_SAMPLE - len(ids)), offset=offset,
                                        with_vectors=[VECTOR_NAME] if VECTOR_NAME else True,
                                        with_payload=[TEXT_FIELD] if TEXT_FIELD else False)
            for p in pts:
                v = p.vector[VECTOR_NAME] if VECTOR_NAME else p.vector
                vecs.append(v); ids.append(p.id)
                if TEXT_FIELD:
                    texts.append(str((p.payload or {}).get(TEXT_FIELD, ""))[:2000])
            if offset is None:
                break
        X_raw, IDS = np.asarray(vecs, dtype=np.float32), np.asarray(ids, dtype=object)
        TEXTS = np.asarray(texts, dtype=object) if TEXT_FIELD else None
        np.savez(cache, X=X_raw, ids=IDS, **({"texts": TEXTS} if TEXTS is not None else {}))
        print(f"scrolled {len(IDS):,} points in {time.time() - t0:.0f}s")
else:
    X_raw, IDS = np.load(NPY_VECTORS).astype(np.float32), np.load(NPY_IDS, allow_pickle=True)
X = pb.normalize(X_raw) if DIST.lower().startswith("cos") else X_raw.astype(np.float32)
N, D = X.shape
QIDX = np.random.default_rng(0).choice(N, min(N_QUERIES, N), replace=False).astype(np.int64)
print(f"{N:,} vectors × {D} dims ({X.nbytes / 2**20:.0f} MB as float32), {len(QIDX):,} queries, distance {DIST}")

## 3. Exact neighbours and embedding health
Exact top-50 neighbours of every query (its own row excluded) are the ground truth for everything below. The health numbers need no labels:

| Number | What a problem looks like |
|---|---|
| `already_normalized` | `False` with a Dot collection: long vectors win regardless of meaning |
| `random_pair_cos_mean` | Close to the neighbours' scores: everything looks similar to everything (boilerplate, a shared direction) |
| `mean_direction_norm` | Above ~0.3: vectors share one dominant direction, which compresses all scores together |
| `exact_duplicate_vectors`, `near_duplicate_queries_pct` | Many: duplicates crowd the top results; they are also the auto-merge candidates |
| `top1pct_hubs_share_of_top10_slots` | Well above 0.01: a few "hub" provisions appear in many unrelated result lists |
| `effective_dim_90pct` | Very low compared with the dimension: the model uses little of its space for our texts |

In [ ]:
t = time.time(); S_TRUE, I_TRUE = pb.exact_topk(X, QIDX, 50); print(f"exact search: {time.time() - t:.1f}s")
health, rand_cos, occ = pb.embedding_health(X_raw, X, S_TRUE, I_TRUE)
display(pd.DataFrame([health]).T.rename(columns={0: "value"}))
RESULTS["embedding_health"] = pd.DataFrame([health])

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
ax[0].hist(rand_cos, 80, alpha=.6, density=True, label="random pairs")
ax[0].hist(S_TRUE[:, 0], 80, alpha=.6, density=True, label="nearest neighbour")
ax[0].hist(S_TRUE[:, K - 1], 80, alpha=.6, density=True, label=f"{K}th neighbour")
ax[0].axvline(MERGE_THRESHOLD, c="k", ls="--", lw=1, label="merge threshold")
ax[0].set(title="Similarity scores: neighbours vs random pairs", xlabel="score"); ax[0].legend(fontsize=8)
ax[1].hist(np.linalg.norm(X_raw, axis=1), 80); ax[1].set(title="Vector norms (before normalization)", xlabel="norm")
ax[2].plot(np.sort(occ)[::-1][:200]); ax[2].set(title=f"Hubness: times each provision appears in a top-10 (top 200)", xlabel="provision rank", ylabel="appearances")
plt.tight_layout(); plt.show()

**Spot check for a domain expert** (only when `TEXT_FIELD` is set): five random provisions with their top-5 neighbours, and the biggest hubs. Judge whether the neighbours are really similar.

In [ ]:
if TEXTS is not None:
    short = lambda s: (s[:220] + "…") if len(s) > 220 else s
    for j in np.random.default_rng(1).choice(len(QIDX), 5, replace=False):
        q = QIDX[j]
        print(f"\n=== QUERY {IDS[q]}: {short(TEXTS[q])}")
        for s, i in zip(S_TRUE[j, :5], I_TRUE[j, :5]):
            print(f"  {s:.3f}  {IDS[i]}: {short(TEXTS[i])}")
    print("\n=== BIGGEST HUBS (appear in many unrelated top-10 lists)")
    for i in np.argsort(-occ)[:5]:
        print(f"  {occ[i]:4d}×  {IDS[i]}: {short(TEXTS[i])}")
else:
    print("TEXT_FIELD not set: no provision text downloaded.")

## 4. Compression benchmark (offline, on the sample)
Each method mimics a Qdrant quantization option: the query stays in float32, stored vectors are compressed. "Rescore" re-ranks an oversampled candidate list with the exact vectors, like `rescore=True` with `oversampling` in Qdrant.

This is an **indicative simulation**: Qdrant's own TurboQuant adds extensions (for example a length renormalization, approximated here by the norm correction), and HNSW adds its own approximation. Section 7 measures the real thing.

In [ ]:
methods = [pb.Float32(), pb.ScalarInt8(0.99), pb.Binary1(), pb.TurboQuantRef(4), pb.TurboQuantRef(2), pb.TurboQuantRef(2, renorm=False)]
if D % 8 != 0:
    methods = [m for m in methods if not isinstance(m, pb.TurboQuantRef)]
    print("dimension not divisible by 8: TurboQuant reference skipped")
try:
    import turbovec  # noqa: F401
    methods += [pb.Turbovec(4), pb.Turbovec(2)]
except ImportError:
    print("turbovec not available: skipped")
rows = []
for m in methods:
    try:
        rows += pb.run_method(m, X, QIDX, I_TRUE[:, :K], k=K, oversampling=(1, 2, 4))
    except Exception as e:
        print(f"{m.name}: failed ({e})")
bench = pd.DataFrame(rows)
display(bench)
RESULTS["compression_benchmark"] = bench

fig, ax = plt.subplots(figsize=(9, 5))
for name, g in bench.groupby("method"):
    ax.plot(g.compression, g[f"recall@{K}"], "o", label=name)
    for _, r in g.iterrows():
        ax.annotate(r.search.replace("rescore, oversampling ", "rs ").replace("no rescore", "raw"), (r.compression, r[f"recall@{K}"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set(xscale="log", xlabel="compression vs float32 (log)", ylabel=f"recall@{K} vs exact", title="Recall vs memory on our provision vectors")
ax.legend(fontsize=7); ax.grid(alpha=.3); plt.show()

## 5. Auto-merge stability
Candidate pairs = each query's exact neighbours scoring at least `MERGE_THRESHOLD − 0.1`. For each compression method we score the same pairs the way a quantized search would (float query against the compressed stored vector) and count decisions that flip at the threshold. With `rescore=True` in Qdrant the final scores are exact, so these flips disappear.

The threshold sweep shows how many pairs would merge at each threshold, and how many pairs sit within ±0.01 of the current one (decisions that any small change can flip).

In [ ]:
A, B, S_PAIR = pb.candidate_pairs(QIDX, S_TRUE, I_TRUE, MERGE_THRESHOLD - 0.1)
near = int((np.abs(S_PAIR - MERGE_THRESHOLD) <= 0.01).sum())
print(f"{len(S_PAIR):,} candidate pairs, {int((S_PAIR >= MERGE_THRESHOLD).sum()):,} at or above the threshold, {near:,} within ±0.01 of it")
flip_rows = []
for m in [pb.ScalarInt8(0.99), pb.TurboQuantRef(4), pb.TurboQuantRef(2), pb.TurboQuantRef(2, renorm=False)]:
    if isinstance(m, pb.TurboQuantRef) and D % 8 != 0:
        continue
    m.build(X)
    flip_rows.append(dict(method=m.name, **pb.merge_flips(S_PAIR, pb.approx_pair_scores(m, X, A, B), MERGE_THRESHOLD)))
flips = pd.DataFrame(flip_rows); display(flips); RESULTS["merge_flips"] = flips

ths = np.round(np.arange(MERGE_THRESHOLD - 0.1, min(MERGE_THRESHOLD + 0.1, 1.0) + 1e-9, 0.005), 3)
sweep = pd.DataFrame(dict(threshold=ths, pairs_that_would_merge=[int((S_PAIR >= t).sum()) for t in ths]))
fig, ax = plt.subplots(figsize=(8, 4)); ax.plot(sweep.threshold, sweep.pairs_that_would_merge, "o-")
ax.axvline(MERGE_THRESHOLD, c="k", ls="--", lw=1); ax.set(xlabel="threshold", ylabel="pairs that would merge", title="Merges vs threshold (exact scores, sampled queries)")
plt.show(); RESULTS["merge_threshold_sweep"] = sweep

## 6. Live search: latency and accuracy (read-only)
The same queries against the production collection in up to three modes: the default search, the same search ignoring quantization (only when the collection is quantized), and an exact search. The overlap with the exact results shows how much the index and the quantization lose; the latencies show what the Qdrant call costs on its own. Embedding the query text is **not** included: time it in the application.

In [ ]:
if SOURCE == "qdrant":
    lq = QIDX[:N_LIVE_QUERIES]
    modes = {"default": None}
    if quant:
        modes["ignore quantization"] = models.SearchParams(quantization=models.QuantizationSearchParams(ignore=True))
        modes["rescore, oversampling 2x"] = models.SearchParams(quantization=models.QuantizationSearchParams(rescore=True, oversampling=2.0))
    modes["exact"] = models.SearchParams(exact=True)
    live_ids, live_ms = {}, {}
    for mode, sp in modes.items():
        res, ms = [], []
        for q in lq:
            t = time.perf_counter()
            r = client.query_points(COLLECTION, query=X_raw[q].tolist(), using=VECTOR_NAME, limit=K + 1,
                                    search_params=sp, with_payload=False, with_vectors=False)
            ms.append((time.perf_counter() - t) * 1000)
            res.append([p.id for p in r.points if p.id != IDS[q]][:K])
        live_ids[mode], live_ms[mode] = res, ms
    ex = live_ids["exact"]
    live_rows = [dict(mode=m, p50_ms=round(float(np.percentile(live_ms[m], 50)), 1), p95_ms=round(float(np.percentile(live_ms[m], 95)), 1),
                      **{f"overlap@{K}_with_exact": round(float(np.mean([len(set(a) & set(b)) / K for a, b in zip(live_ids[m], ex)])), 4)})
                 for m in modes]
    live_df = pd.DataFrame(live_rows); display(live_df); RESULTS["live_search"] = live_df
    print("Reading it: overlap well below 1.0 in 'default' but close to 1.0 in 'ignore quantization' = quantization loses results "
          "(turn on rescore); low in both = the HNSW index loses results (raise hnsw_ef or m). Latency includes network round trips.")

## 7. Optional: real Qdrant quantization, in a temporary local server
Starts Qdrant `QDRANT_VERSION` inside this Colab runtime (nothing touches production), loads the sample into one collection per quantization option, and measures recall against the exact neighbours with and without rescoring. TurboQuant needs Qdrant 1.18 or later. If a configuration is not accepted by that version, it is reported and skipped.

In [ ]:
if RUN_LOCAL_QDRANT:
    import requests, tarfile, urllib.request
    from qdrant_client import QdrantClient, models
    LOCAL = "http://localhost:6333"
    exe = "/tmp/qdrant/qdrant"
    if not os.path.exists(exe):
        url = f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_VERSION}/qdrant-x86_64-unknown-linux-gnu.tar.gz"
        os.makedirs("/tmp/qdrant", exist_ok=True)
        try:
            urllib.request.urlretrieve(url, "/tmp/qdrant/q.tgz"); tarfile.open("/tmp/qdrant/q.tgz").extractall("/tmp/qdrant")
        except Exception as e:
            print(f"Could not download {url} ({e}). Pick an existing release at github.com/qdrant/qdrant/releases and set QDRANT_VERSION.")
    if os.path.exists(exe):
        env = dict(os.environ, QDRANT__STORAGE__STORAGE_PATH="/tmp/qdrant/storage", QDRANT__TELEMETRY_DISABLED="true")
        if "qproc" not in globals() or qproc.poll() is not None:
            qproc = subprocess.Popen([exe], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            for _ in range(60):
                try:
                    requests.get(LOCAL + "/readyz", timeout=1); break
                except Exception:
                    time.sleep(1)
        print("local Qdrant:", requests.get(LOCAL).json().get("version"))
        lc = QdrantClient(url=LOCAL, timeout=300)
        dist = "Cosine" if DIST.lower().startswith("cos") else "Dot"
        configs = {"float32": None,
                   "scalar int8": {"scalar": {"type": "int8", "quantile": 0.99, "always_ram": True}},
                   "binary 1-bit": {"binary": {"always_ram": True}},
                   "turbo 4-bit": {"turbo": {"bits": "bits4"}},
                   "turbo 2-bit": {"turbo": {"bits": "bits2"}}}
        local_rows = []
        for label, qc in configs.items():
            name = "bench_" + label.replace(" ", "_").replace("-", "")
            requests.delete(f"{LOCAL}/collections/{name}")
            body = {"vectors": {"size": D, "distance": dist}}
            if qc:
                body["quantization_config"] = qc
            r = requests.put(f"{LOCAL}/collections/{name}", json=body)
            if not r.ok:
                print(f"{label}: not accepted by this Qdrant version ({r.status_code}: {r.text[:200]})"); continue
            t = time.time()
            lc.upload_collection(name, vectors=X_raw, ids=list(range(N)), batch_size=512, parallel=2, wait=True)
            for _ in range(600):
                ci = lc.get_collection(name)
                if str(ci.status).lower().endswith("green") and (ci.indexed_vectors_count or 0) >= 0.99 * N:
                    break
                time.sleep(2)
            build_s = time.time() - t
            for rescore in ([False, True] if qc else [False]):
                sp = models.SearchParams(quantization=models.QuantizationSearchParams(rescore=rescore, oversampling=2.0 if rescore else 1.0)) if qc else None
                got, ms = [], []
                for q in QIDX[:300]:
                    t = time.perf_counter()
                    res = lc.query_points(name, query=X_raw[q].tolist(), limit=K + 1, search_params=sp, with_payload=False)
                    ms.append((time.perf_counter() - t) * 1000)
                    got.append([p.id for p in res.points if p.id != q][:K])
                got = np.array([g + [-1] * (K - len(g)) for g in got])
                local_rows.append(dict(config=label, rescore=rescore, build_and_index_s=round(build_s, 1),
                                       p50_ms=round(float(np.percentile(ms, 50)), 2),
                                       **{f"recall@{K}": round(pb.recall_at_k(got, I_TRUE[:300, :K], K), 4),
                                          "top1_found_in_top1": round(pb.r1_at_k(got, I_TRUE[:300, :K], 1), 4)}))
        local_df = pd.DataFrame(local_rows); display(local_df); RESULTS["local_qdrant"] = local_df
        print("Memory per vector, float32:", 4 * D, "bytes; scalar int8 ≈", D, "; turbo 4-bit ≈", D // 2, "; turbo 2-bit ≈", D // 4, "; binary ≈", D // 8)

## 8. Optional: labelled evaluation
Upload CSVs prepared by domain experts (Files pane on the left), using **Qdrant point ids**:
* `similar_labels.csv` with columns `query_id, relevant_id`: provisions that should come back as similar.
* `merge_labels.csv` with columns `id_a, id_b, same`: pairs marked 1 (same provision, should merge) or 0 (different).

Labelled ids must be inside the loaded sample.

In [ ]:
row_of = {str(i): r for r, i in enumerate(IDS)}
if os.path.exists(SIMILAR_LABELS_CSV):
    lab = pd.read_csv(SIMILAR_LABELS_CSV, dtype=str)
    rel = {}
    for qa, rb in zip(lab.query_id, lab.relevant_id):
        if qa in row_of and rb in row_of:
            rel.setdefault(row_of[qa], set()).add(row_of[rb])
    lq_rows = np.array(sorted(rel), dtype=np.int64)
    _, I_lab = pb.exact_topk(X, lq_rows, K)
    lab_rows = [dict(method="float32 exact", **{f"label_recall@{K}": round(pb.recall_from_labels(I_lab, lq_rows, rel, K), 4)})]
    for m in [pb.ScalarInt8(0.99), pb.TurboQuantRef(4), pb.TurboQuantRef(2)]:
        m.build(X); _, Im = m.search(X, lq_rows, K)
        lab_rows.append(dict(method=m.name, **{f"label_recall@{K}": round(pb.recall_from_labels(Im, lq_rows, rel, K), 4)}))
    lab_df = pd.DataFrame(lab_rows); display(lab_df); RESULTS["labelled_recall"] = lab_df
    print(f"{len(rel)} labelled queries. Low label recall even for 'float32 exact' = the embeddings, not the index or compression, are the problem.")
else:
    print(f"{SIMILAR_LABELS_CSV} not found: skipped")
if os.path.exists(MERGE_LABELS_CSV):
    ml = pd.read_csv(MERGE_LABELS_CSV, dtype={"id_a": str, "id_b": str})
    ml = ml[ml.id_a.isin(row_of) & ml.id_b.isin(row_of)]
    a = np.array([row_of[i] for i in ml.id_a]); b = np.array([row_of[i] for i in ml.id_b])
    sc = np.einsum("nd,nd->n", X[a], X[b])
    pr = pd.DataFrame(pb.merge_pr_curve(sc, ml.same.astype(int).values, np.round(np.arange(0.70, 1.0, 0.01), 2)))
    display(pr); RESULTS["merge_precision_recall"] = pr
    fig, ax = plt.subplots(figsize=(8, 4)); ax.plot(pr.threshold, pr.precision, "o-", label="precision"); ax.plot(pr.threshold, pr.recall, "o-", label="recall")
    ax.axvline(MERGE_THRESHOLD, c="k", ls="--", lw=1); ax.set(xlabel="threshold", title=f"Auto-merge on {len(ml)} labelled pairs"); ax.legend(); plt.show()
else:
    print(f"{MERGE_LABELS_CSV} not found: skipped")

## 9. Save the results
Writes one CSV per table to `results/` (numbers only: no vectors, no provision text) and zips them for download. Bring them back to the team, not the executed notebook if it shows provision text.

In [ ]:
os.makedirs("results", exist_ok=True)
for name, df in RESULTS.items():
    df.to_csv(f"results/provision_{name}.csv", index=False)
import shutil
shutil.make_archive("provision_benchmark_results", "zip", "results")
print("saved:", sorted(os.listdir("results")))
try:
    from google.colab import files
    files.download("provision_benchmark_results.zip")
except Exception:
    pass